# Covered Linkway / Shelter — GeoSAM-TopoLoRA

**Architecture**: SAM2.1 Hiera Large (frozen) + LoRA (rank=16) on the QKV blocks + UNet decoder, trained with weighted BCE + IoU + **clDice** topology loss (the *TopoLoRA* recipe from Seglab), augmented with the GeoSAM-style mask-supervised fine-tuning on aerial tiles.

**Pipeline**

| Step | Cell |
|------|------|
| 0. Path configuration | 2 |
| 1. Regenerate the train/val grid GPKG | 4 |
| 2. Rebuild the training masks from JSON + stitch one big mask | 6 |
| 3. Build the SAM2 + LoRA + UNet model | 8 |
| 4. List existing checkpoints (training can be skipped) | 10 |
| 5. Training (progress report every 10 tiles for <=1000 tiles / every 100 tiles for >1000 tiles) | 12 |
| 6. Full-image mosaic inference (GPU/CPU parallel) + building / footpath constraints | 14 |
| 7. Morphological post-processing + vectorised output TIF/GPKG/SHP | 16 |
| 8. Visualisation | 18 |

Environment: `conda activate sam2`

## 0. Path configuration (the only cell to edit)

In [ ]:
import os, sys

# ── Project paths ───────────────────────────────────────────────────────────────
PROJECT_DIR = r"D:\Claude\GeoSAM-TopoLoRA"
SAM2_REPO   = r"D:\Claude\Meta-SAM2\models\sam2-main"
CLP_DIR     = r"D:\Claude\Meta-SAM2"   # contains the SAM2-UNet engine

for _p in (PROJECT_DIR, SAM2_REPO, CLP_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── SAM2.1 backbone ────────────────────────────────────────────────────────
SAM2_CHECKPOINT = os.path.join(SAM2_REPO, r"checkpoints\sam2.1_hiera_large.pt")
SAM2_CONFIG     = "configs/sam2.1/sam2.1_hiera_l.yaml"

# ── Data directories ───────────────────────────────────────────────────────────────
DATA_BASE  = os.path.join(PROJECT_DIR, "covered Linkway")
IMAGES_DIR = os.path.join(DATA_BASE, "images")   # train/ val/
JSON_DIR   = os.path.join(DATA_BASE, "json")     # train/ val/
MASKS_DIR  = os.path.join(DATA_BASE, "masks")    # train/ val/  (generated)
GRID_DIR   = os.path.join(DATA_BASE, "grid")
GRID_CSV   = os.path.join(GRID_DIR, "tiles_meta.csv")

# ── Input / output TIF ─────────────────────────────────────────────────────────
INPUT_TIF  = os.path.join(DATA_BASE, "SG_google_map_4000_03m.tif")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
BIG_MASK_TIF      = os.path.join(OUTPUT_DIR, "labels_big_mask.tif")
LINKWAY_TIF       = os.path.join(OUTPUT_DIR, "covered_linkway_mask.tif")
LINKWAY_TIF_POST  = os.path.join(OUTPUT_DIR, "covered_linkway_mask_post.tif")

# ── Vector constraints ───────────────────────────────────────────────────────────────
BOUNDARY_SHP    = os.path.join(PROJECT_DIR, r"shp\SUB_SG_polygon\SUB_SG_polygon.shp")
BUILDING_GPKG   = r"C:\Users\City Syntax Lab\Desktop\SOLWEIG_GPU\data\1-SG data\Shp\SG_Building\SG_Building_SVY21.gpkg"
FOOTPATH_GPKG   = r"C:\Users\City Syntax Lab\Desktop\Covered Linkways\1-data\1-SG\Footpath_Mar2026\Footpath.gpkg"
USE_FOOTPATH    = True       # set False if you only want the building filter
FOOTPATH_BUFFER = 2.0        # metres

# ── Checkpoints ────────────────────────────────────────────────────────────
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Training / inference hyper-parameters ───────────────────────────────────────────────────────
TILE_SIZE      = 1024
OVERLAP        = 128
BATCH_SIZE     = 2
EPOCHS         = 20
LR             = 1e-4
POS_WEIGHT     = 50.0
CLDICE_WEIGHT  = 0.3      # tuned for linkway (wider than vessels)
LORA_RANK      = 16
LORA_ALPHA     = 32.0
DEVICE         = "cuda"

THRESHOLD      = 0.5
CLOSING_RADIUS = 5
MIN_AREA_M2    = 4.0
RGB_BANDS      = (1, 2, 3)

print("Config OK")
print(f"  Input TIF      : {INPUT_TIF}  (exists={os.path.exists(INPUT_TIF)})")
print(f"  Boundary SHP   : {BOUNDARY_SHP}  (exists={os.path.exists(BOUNDARY_SHP)})")
print(f"  Building GPKG  : {os.path.exists(BUILDING_GPKG)}")
print(f"  Footpath GPKG  : {os.path.exists(FOOTPATH_GPKG)} (use={USE_FOOTPATH})")
print(f"  SAM2 weights   : {os.path.exists(SAM2_CHECKPOINT)}")
print(f"  Outputs        : {OUTPUT_DIR}")

## 1. Regenerate the train/val grid files (GPKG + CSV)

Rebuild the grid from the PNG file names in `images/{train,val}` (`tile_x{col}_y{row}.png`) and the geotransform of the input TIF.

In [ ]:
import importlib, gsltl_pipeline as g
importlib.reload(g)

grid_info = g.generate_grids_from_labelled_tiles(
    input_tif   = INPUT_TIF,
    images_dir  = IMAGES_DIR,
    grid_dir    = GRID_DIR,
    splits      = ("train", "val"),
    tile_size   = TILE_SIZE,
    overlap     = OVERLAP,
    crs_epsg    = 3414,
    verbose     = True,
)

## 2. JSON → mask generation + big mask stitching

(a) Convert the LabelMe JSON files to binary mask PNGs (empty annotation = all black)

(b) Stitch all train+val masks into one big GeoTIFF of the same size as the input TIF; overlapping white pixels are OR-combined.

In [ ]:
import importlib, gsltl_pipeline as g
importlib.reload(g)

# (a) JSON → PNG mask
stats = g.generate_masks_from_json(
    images_dir = IMAGES_DIR,
    json_dir   = JSON_DIR,
    masks_dir  = MASKS_DIR,
    splits     = ("train", "val"),
    verbose    = True,
)
for split, s in stats.items():
    print(f"  [{split}] {s['written']}/{s['total']} written, {s['empty']} empty")

# (b) stitch one big mask
big = g.stitch_labeled_masks_to_big_mask(
    masks_dir  = MASKS_DIR,
    input_tif  = INPUT_TIF,
    output_tif = BIG_MASK_TIF,
    splits     = ("train", "val"),
    verbose    = True,
)
print(f"\nBig label mask: {BIG_MASK_TIF}  (placed {big['placed']}/{big['total']} tiles)")

## 3. Build the model (SAM2-UNet + LoRA)

SAM2.1 Hiera Large encoder frozen + LoRA rank=16 injected into QKV + trainable UNet decoder.

**TopoLoRA**: clDice topology loss added on top of BCE+IoU with weight `CLDICE_WEIGHT=0.3` (reduced because linkways are much wider than vessels).

In [ ]:
import torch, importlib, gsltl_pipeline as g
importlib.reload(g)

if DEVICE == "cuda" and not torch.cuda.is_available():
    print("[warn] CUDA not available, falling back to CPU")
    DEVICE = "cpu"
elif DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}  | "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

print("\nBuilding SAM2-UNet+LoRA model...")
model = g.SAM2UNetLoRA(
    sam2_repo       = SAM2_REPO,
    sam2_checkpoint = SAM2_CHECKPOINT,
    sam2_config     = SAM2_CONFIG,
    lora_rank       = LORA_RANK,
    lora_alpha      = LORA_ALPHA,
).to(DEVICE)

print(g.count_trainable_params(model))

## 4. List existing checkpoints (training can be skipped)

In [ ]:
import gsltl_pipeline as g
ckpts = g.list_checkpoints(CHECKPOINT_DIR)

# To load a checkpoint directly, uncomment:
# best_ckpt = ckpts[-1]
# model = g.load_checkpoint(model, best_ckpt, device=DEVICE)

## 5. Training

Progress: one summary line per epoch, with an inner tqdm bar at batch level.
Total training samples: train 1616 tiles (>1000), so batch reports are issued every 100 steps.

In [ ]:
import gsltl_pipeline as g

train_loader, val_loader = g.get_dataloaders(
    images_dir  = IMAGES_DIR,
    masks_dir   = MASKS_DIR,
    batch_size  = BATCH_SIZE,
    img_size    = TILE_SIZE,
    num_workers = 0,
)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

best_ckpt = g.train(
    model          = model,
    train_loader   = train_loader,
    val_loader     = val_loader,
    epochs         = EPOCHS,
    lr             = LR,
    weight_decay   = 5e-4,
    cldice_weight  = CLDICE_WEIGHT,
    pos_weight     = POS_WEIGHT,
    save_dir       = CHECKPOINT_DIR,
    device         = DEVICE,
    verbose        = True,
)
print(f"\nBest checkpoint: {best_ckpt}")

## 6. Full-image mosaic inference + GIS constraints

- `use_parallel=True`: GPU batch + 4-stage pipeline (reader / GPU / post / writer) — recommended.
- `use_parallel=False`: sequential inference (use without a GPU or with very little VRAM).
- The building footprint **must** be provided (used as an exclusion mask).
- The footpath layer is optional (with `USE_FOOTPATH=True`, only connected components intersecting the footpath buffer are kept).

In [ ]:
import importlib, gsltl_pipeline as g, torch
importlib.reload(g)

USE_PARALLEL = (DEVICE == "cuda")        # parallel on GPU, sequential automatically on CPU
info = g.run_mosaic_inference(
    input_tif         = INPUT_TIF,
    model             = model,
    output_tif        = LINKWAY_TIF,
    boundary_shp      = BOUNDARY_SHP,
    building_gpkg     = BUILDING_GPKG,
    footpath_gpkg     = FOOTPATH_GPKG if USE_FOOTPATH else None,
    footpath_buffer_m = FOOTPATH_BUFFER,
    tile_size         = TILE_SIZE,
    overlap           = OVERLAP,
    threshold         = THRESHOLD,
    rgb_bands         = RGB_BANDS,
    use_tta           = True,
    device            = DEVICE,
    batch_size        = 8 if DEVICE == "cuda" else 1,
    use_parallel      = USE_PARALLEL,
    use_amp_inference = (DEVICE == "cuda"),
    verbose           = True,
)
print(f"\nMosaic TIF: {info['output_tif']}")
print(f"  total tiles = {info['total_tiles']:,} | valid = {info['valid_tiles']:,}")

## 7. Morphological post-processing + vectorisation

- Closing fills the small gaps inside the linkways
- Fragments smaller than `MIN_AREA_M2` are removed
- Outputs GeoTIFF + GPKG + SHP

In [ ]:
import importlib, gsltl_pipeline as g, rasterio, numpy as np
importlib.reload(g)

with rasterio.open(LINKWAY_TIF) as src:
    mask = src.read(1)
    profile = src.profile.copy()
    pix = abs(src.transform.a)

min_area_px = max(10, int(MIN_AREA_M2 / (pix ** 2)))
print(f"pixel_size={pix:.2f} m  |  min_area_px={min_area_px}")

mask_post = g.connectivity_postprocess(
    mask           = mask,
    closing_radius = CLOSING_RADIUS,
    min_area_px    = min_area_px,
    verbose        = True,
)

g.save_mask_tif(mask_post, LINKWAY_TIF_POST, profile)

gpkg, shp = g.vectorise_mosaic(
    mosaic_tif  = LINKWAY_TIF_POST,
    output_dir  = OUTPUT_DIR,
    name        = "covered_linkway",
    min_area_m2 = MIN_AREA_M2,
    verbose     = True,
)
print(f"\nFinal outputs:")
print(f"  TIF (raw)  : {LINKWAY_TIF}")
print(f"  TIF (post) : {LINKWAY_TIF_POST}")
print(f"  GPKG       : {gpkg}")
print(f"  SHP        : {shp}")

## 8. Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, rasterio
from PIL import Image as PILImage

with rasterio.open(INPUT_TIF) as src:
    ovr = max(1, src.width // 1000)
    rgb = src.read(list(RGB_BANDS),
                   out_shape=(3, src.height // ovr, src.width // ovr)
                  ).transpose(1, 2, 0)
    rgb = (rgb - rgb.min()) / (rgb.ptp() + 1e-6)

with rasterio.open(LINKWAY_TIF_POST) as src:
    mfull = src.read(1)

h, w = rgb.shape[:2]
msk_small = np.array(
    PILImage.fromarray(mfull * 255).resize((w, h), PILImage.NEAREST)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(rgb);                axes[0].set_title("Google map"); axes[0].axis("off")
axes[1].imshow(msk_small, cmap="Reds")
axes[1].set_title(f"Linkway mask  px={int(mfull.sum()):,}");          axes[1].axis("off")
overlay = rgb.copy(); overlay[msk_small > 0] = [1, 0.2, 0.2]
axes[2].imshow(overlay);            axes[2].set_title("Overlay");   axes[2].axis("off")
plt.suptitle("Covered Linkway — GeoSAM-TopoLoRA", y=1.01)
plt.tight_layout()
preview = os.path.join(OUTPUT_DIR, "preview.png")
plt.savefig(preview, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {preview}")

## Appendix: tuning notes

| Goal | Parameter |
|---|---|
| Higher recall | `THRESHOLD` ↓ 0.3 |
| Fewer false positives | `THRESHOLD` ↑ 0.6 |
| Fill gaps inside linkways | `CLOSING_RADIUS` ↑ |
| Remove fragments | `MIN_AREA_M2` ↑ |
| Class imbalance | `POS_WEIGHT` 30~100 |
| Topological connectivity | `CLDICE_WEIGHT` 0.2~0.5 (linkways are wide, start at 0.3) |

**Notes on parallel mosaic inference**
- GPU path: 4-stage pipeline (reader / GPU batch / post-proc / writer) + AMP, GPU utilisation 70-90%.
- CPU path: sequential inference, for validation on machines without a GPU.

**Progress report rules**
- total tiles < 1000 → report every 10 tiles
- total tiles ≥ 1000 → report every 100 tiles
(applies in `generate_masks_from_json`, `stitch_labeled_masks_to_big_mask`, `run_tile_inference`, `run_island_full_inference*` and the footpath post-processing.)